# Prioritas Wilayah

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "hasil_prioritas_wilayah"
OUT.mkdir(parents=True, exist_ok=True)
polusi_path = ROOT / "hasil_analisis_moniq" / "06_ringkasan_polusi_wilayah_untuk_falisha.csv"
kesehatan_path = ROOT / "hasil_analisis_kesehatan" / "04_perbandingan_wilayah.csv"
if not (polusi_path.is_file() and kesehatan_path.is_file()):
    raise FileNotFoundError("Jalankan notebook EDA kualitas udara dan EDA kesehatan terlebih dahulu.")
polusi = pd.read_csv(polusi_path)
kesehatan = pd.read_csv(kesehatan_path)
print("Polusi:", polusi.shape, "Kesehatan:", kesehatan.shape)


In [ ]:
for df in (polusi, kesehatan):
    for col in ["provinsi", "kota_kabupaten"]:
        df[col] = df[col].astype("string").str.strip().str.casefold()


In [ ]:
gabung = polusi.merge(kesehatan, on=["provinsi", "kota_kabupaten"], how="outer", indicator=True, validate="one_to_one")
print(gabung["_merge"].value_counts())
display(gabung.loc[gabung["_merge"] != "both", ["provinsi", "kota_kabupaten", "_merge"]])


## Beban kasus harian

In [ ]:
gabung["beban_pernapasan_harian_tercatat"] = (gabung["resp_admissions_under5"] + gabung["resp_admissions_over60"]) / gabung["jumlah_hari"].replace(0, np.nan)
layak = gabung.loc[(gabung["_merge"] == "both") & gabung["pm25_ug_m3"].notna() & gabung["beban_pernapasan_harian_tercatat"].notna()].copy()
print("Wilayah dapat dibandingkan:", len(layak))


## Kategori prioritas

In [ ]:
if layak.empty:
    raise ValueError("Tidak ada wilayah dengan data PM2.5 dan kasus yang cocok.")
median_pm25 = layak["pm25_ug_m3"].median()
median_kasus = layak["beban_pernapasan_harian_tercatat"].median()
layak["kategori_prioritas"] = np.select([
    (layak["pm25_ug_m3"] >= median_pm25) & (layak["beban_pernapasan_harian_tercatat"] >= median_kasus),
    layak["pm25_ug_m3"] >= median_pm25,
    layak["beban_pernapasan_harian_tercatat"] >= median_kasus,
], ["PM2.5 tinggi & kasus tinggi", "PM2.5 tinggi saja", "Kasus tinggi saja"], default="Keduanya relatif rendah")
display(layak["kategori_prioritas"].value_counts())


In [ ]:
layak["persentil_pm25"] = layak["pm25_ug_m3"].rank(pct=True)
layak["persentil_beban_kasus"] = layak["beban_pernapasan_harian_tercatat"].rank(pct=True)
layak["skor_prioritas_eksploratif"] = 100 * (layak["persentil_pm25"] + layak["persentil_beban_kasus"]) / 2
layak = layak.sort_values("skor_prioritas_eksploratif", ascending=False)
layak["peringkat_eksploratif"] = range(1, len(layak) + 1)
display(layak[["kota_kabupaten", "pm25_ug_m3", "beban_pernapasan_harian_tercatat", "kategori_prioritas", "skor_prioritas_eksploratif"]].head(10))


## Matriks wilayah

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for kategori, sub in layak.groupby("kategori_prioritas"):
    ax.scatter(sub["pm25_ug_m3"], sub["beban_pernapasan_harian_tercatat"], label=kategori, alpha=.8)
ax.axvline(median_pm25, ls="--", alpha=.5)
ax.axhline(median_kasus, ls="--", alpha=.5)
ax.set(xlabel="PM2.5 rerata (µg/m³)", ylabel="Kasus pernapasan per hari catatan RS", title="Matriks prioritas eksploratif")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "grafik_matriks_prioritas.png", dpi=160)
plt.show()


In [ ]:
layak.to_csv(OUT / "01_priority_areas_wilayah.csv", index=False)
gabung.to_csv(OUT / "02_audit_kecocokan_wilayah.csv", index=False)
kolom = ["provinsi", "kota_kabupaten", "pm25_ug_m3", "beban_pernapasan_harian_tercatat", "skor_prioritas_eksploratif", "kategori_prioritas", "peringkat_eksploratif", "jumlah_stasiun", "jumlah_rumah_sakit", "jumlah_hari"]
layak[kolom].to_csv(OUT / "03_tabel_dashboard_priority_areas.csv", index=False)
layak.groupby("kategori_prioritas", as_index=False).agg(jumlah_wilayah=("kota_kabupaten", "size")).to_csv(OUT / "04_rekap_kategori.csv", index=False)
print("Tersimpan di:", OUT)
